# Paderborn Bearing Fault Diagnosis — Main Pipeline

A complete end-to-end workflow for bearing fault classification using the Paderborn
University bearing dataset. This notebook ties together the four supporting modules
(`00_download_dataset`, `01_data_loader`, `02_dsp_features`, `03_ml_classification`)
into a single, reproducible pipeline.

**Dataset:** Paderborn University Bearing Data Center  
**Signals:** Phase currents (64 kHz), vibration (64 kHz), operating conditions (4 kHz)  
**Task:** 3-class fault classification — Healthy / Outer Race Fault / Inner Race Fault

In [257]:
# =========================================================
# 0. Imports & Configuration
# =========================================================

# --- Standard library ---
import os
import sys
import glob
import importlib.util
import warnings
from datetime import datetime
from pathlib import Path

# --- Third-party core ---
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- Visualisation ---
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import stft

# --- scikit-learn ---
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

# --- Other ML ---
import shap

# --- Signal processing ---
import pywt

# --- Dynamic loading of numbered-prefix modules ---
# Jupyter's working directory is the notebook directory, so Path('.') is reliable.
_BASE_DIR = Path('.').resolve()

def _load_module(name: str, filename: str):
    """Load a .py module whose filename has a numeric prefix."""
    path = _BASE_DIR / filename
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

data_loader   = _load_module("data_loader",      "01_data_loader.py")
downloader    = _load_module("download_dataset",  "00_download_dataset.py")
dsp_mod       = _load_module("dsp_features",      "02_dsp_features.py")
ml_mod        = _load_module("ml_classification", "03_ml_classification.py")
plot_style    = _load_module("plot_style",        "utils/plot_style.py")

from data_loader import (
    load_mat_file, load_dataset, calc_characteristic_frequencies,
    OPERATING_CONDITIONS,
)
from download_dataset import ensure_data, MINIMAL_SET, FULL_SET
from dsp_features import (
    time_domain_features, frequency_domain_features,
    wavelet_packet_features, envelope_analysis, extract_all_features,
    extract_features_from_bearing,
)
from ml_classification import (
    TraditionalMLPipeline,
    EXPERIMENT_ARTIFICIAL_TO_REAL, EXPERIMENT_REAL_CV,
)
from plot_style import (
    apply_style, FigSize,
    CMAP,     C1,  C2,  C3,  FAULT_COLORS,
    CMAP_DMG, D1,  D2,  D3,  FAULT_COLORS_DMG,
)

warnings.filterwarnings("ignore")
apply_style()   # sets whitegrid theme, palette colour cycle, default figure size

In [258]:
# --- Constants (all tuneable values live here) ---
RANDOM_STATE = 42      # global seed
TEST_SIZE    = 0.20    # held-out test fraction
N_SPLITS     = 5       # cross-validation folds

# Signal-display
N_SHOW_SECONDS    = 0.1     # seconds of signal to display in time-domain plots

# Spectrum ranges — consistent across FFT and STFT for the same signal type
FFT_CURRENT_MAX_HZ    = 500      # fault sidebands and 50 Hz harmonics live below 500 Hz
FFT_VIB_MAX_HZ        = 10000    # bearing resonance bands typically sit in 2–10 kHz
STFT_CURRENT_MAX_HZ   = 500      # matches FFT window for current
STFT_VIB_MAX_HZ       = 10000    # matches FFT window for vibration
ENVELOPE_BAND         = (500, 10000)   # bandpass before Hilbert envelope (vibration)
ENVELOPE_CURRENT_BAND = (50, 5000)
ENVELOPE_MAX_HZ       = 300

# STFT
STFT_NPERSEG  = 2048
STFT_NOVERLAP = 1536

# Dataset selection — controls which bearings are downloaded AND iterated.
#   MINIMAL_SET : 15 bearings, real damage only  (~2.4 GB)
#   FULL_SET    : 32 bearings, includes artificial damage  (~5 GB)
BEARING_SET = MINIMAL_SET   # swap to FULL_SET to use all bearings

# Operating condition filter.
#   None  → use all four conditions (N15_M07_F10, N09_M07_F10, N15_M01_F10, N15_M07_F04)
#           ~1 200 signals total; absolute-frequency features are normalised to
#           speed-invariant orders in Section 3f-ii.
#   str   → restrict to a single condition, e.g. 'N15_M07_F10'
SETTING_FILTER  = None
USE_CURRENT     = True
USE_VIBRATION   = True

# DEV_MODE: cap signals loaded per bearing folder for fast iteration.
# Each folder has up to 20 files per condition (80 across all 4); set to None for full run.
N_SIGNALS_PER_BEARING = None   # e.g. 2 x 15 = 30 signals; set None for all

# Explainability
SHAP_SAMPLE_SIZE = 200   # rows of X_test passed to TreeExplainer (speed vs. detail)

PLOTS_DIR = _BASE_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

print(f"Pipeline initialised   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Base directory         {_BASE_DIR}")
print(f"Plots directory        {PLOTS_DIR}")
print(f"BEARING_SET            {len(BEARING_SET)} bearings  "
      f"({'MINIMAL_SET' if BEARING_SET is MINIMAL_SET else 'FULL_SET'})")
print(f"SETTING_FILTER         {SETTING_FILTER or 'ALL (4 conditions)'}")
print(f"N_SIGNALS_PER_BEARING  {N_SIGNALS_PER_BEARING}  (None = all)")

Pipeline initialised   2026-03-11 19:30:28
Base directory         C:\8. ds
Plots directory        C:\8. ds\plots
BEARING_SET            15 bearings  (MINIMAL_SET)
SETTING_FILTER         ALL (4 conditions)
N_SIGNALS_PER_BEARING  None  (None = all)


## 1. Data Acquisition

`ensure_data` checks whether the required `.mat` files are already present and
downloads only the bearings listed in `MINIMAL_SET` if they are missing.
The returned path is the root directory that contains one sub-folder per bearing code.

In [259]:
# =========================================================
# 1. Data Acquisition
# =========================================================

mat_dir = ensure_data(BEARING_SET)
print(f"\nData directory: {mat_dir}")

# Locate .mat files for a healthy bearing (K001) as the exploration example
mat_files = glob.glob(str(mat_dir / 'K001' / '*.mat'))

if not mat_files:
    raise FileNotFoundError(
        f"No .mat files found under {mat_dir / 'K001'}. "
        "Re-run ensure_data or check the download path."
    )

EXAMPLE_FILE = mat_files[0]
print(f"Example file  : {os.path.basename(EXAMPLE_FILE)}")
print(f"Files available: {len(mat_files)}")

Dataset ready: all 15 bearings found in C:\8. ds\paderborn_data\mat

Data directory: C:\8. ds\paderborn_data\mat
Example file  : N09_M07_F10_K001_1.mat
Files available: 80


## 2. Exploratory Data Analysis

Load one representative `.mat` file and inspect its metadata and signal statistics.
This confirms the data loaded correctly and surfaces the operating conditions
(speed, torque, force) that will become important for stratification later.
Bearing characteristic frequencies are calculated from the measured shaft speed
and will be used to interpret spectral plots in Section 7.

In [260]:
# =========================================================
# 2. Exploratory Data Analysis
# =========================================================

sig = load_mat_file(EXAMPLE_FILE)

print(f"Bearing        : {sig.bearing_code}")
print(f"Condition      : {sig.label_name}  (class {sig.label_3class})")
print(f"Damage origin  : {sig.damage_origin}")
print(f"Operating setting: {sig.setting}")

print("\n--- Signal Info ---")
print(f"  Sampling rate : {sig.fs} Hz")
print(f"  Duration      : {sig.duration} s")
print(f"  Current 1   shape={sig.phase_current_1.shape}  "
      f"RMS={np.sqrt(np.mean(sig.phase_current_1**2)):.4f}")
print(f"  Current 2   shape={sig.phase_current_2.shape}  "
      f"RMS={np.sqrt(np.mean(sig.phase_current_2**2)):.4f}")
print(f"  Vibration   shape={sig.vibration.shape}  "
      f"RMS={np.sqrt(np.mean(sig.vibration**2)):.4f}")

print("\n--- Operating Conditions ---")
print(f"  Speed       : {sig.speed.mean():.1f} rpm")
print(f"  Torque      : {sig.torque.mean():.3f} Nm")
print(f"  Force       : {sig.force.mean():.1f} N")
print(f"  Temperature : {sig.temperature.mean():.1f} °C")

Bearing        : K001
Condition      : Healthy  (class 0)
Damage origin  : healthy
Operating setting: N09_M07_F10

--- Signal Info ---
  Sampling rate : 64000 Hz
  Duration      : 4.0 s
  Current 1   shape=(256823,)  RMS=1.6567
  Current 2   shape=(256823,)  RMS=1.6681
  Vibration   shape=(256823,)  RMS=0.3676

--- Operating Conditions ---
  Speed       : 899.7 rpm
  Torque      : 1.200 Nm
  Force       : 1025.9 N
  Temperature : 46.9 °C


In [261]:
# --- 2a. Bearing Characteristic Frequencies ---
# Derived from geometry; used as reference markers in spectral plots.

char_freqs = calc_characteristic_frequencies(sig.speed.mean())

print("Bearing characteristic frequencies:")
for name, f in char_freqs.items():
    print(f"  {name:6s}: {f:.2f} Hz")

Bearing characteristic frequencies:
  shaft_freq: 15.00 Hz
  BPFO  : 45.80 Hz
  BPFI  : 74.16 Hz
  BSF   : 29.94 Hz
  FTF   : 5.72 Hz


## 3. Preprocessing Pipeline — DSP Feature Extraction

Raw time-series are transformed into a fixed-length feature vector per segment.
Four complementary feature groups capture different fault signatures:

| Group | Domain | Captures |
|---|---|---|
| Time-domain | Time | Amplitude statistics (RMS, kurtosis, crest factor …) |
| Frequency-domain | Frequency | Spectral centroid, bandwidth, harmonic ratios |
| WPD (Wavelet Packet) | Time-frequency | Sub-band energy distribution |
| Envelope | Frequency (demodulated) | Fault impulse repetition rates |

Each feature group is demonstrated on the example signal; the final cell runs
`extract_all_features` which combines all groups into one vector.

In [262]:
# =========================================================
# 3. Preprocessing Pipeline
# =========================================================

# --- 3a. Time-Domain Features (Phase Current 1) ---
td = time_domain_features(sig.phase_current_1)
print("Time-domain features:")
for k, v in td.items():
    print(f"  {k:<25}: {v:.6f}")

Time-domain features:
  mean                     : 0.023506
  std                      : 1.656556
  rms                      : 1.656722
  peak                     : 2.877041
  peak_to_peak             : 5.730664
  skewness                 : 0.006718
  kurtosis                 : 1.506477
  crest_factor             : 1.736586
  shape_factor             : 1.112840
  impulse_factor           : 1.932544
  clearance_factor         : 2.119184
  energy                   : 704909.462866
  entropy                  : 2.150155


In [263]:
# --- 3b. Frequency-Domain Features ---
fd = frequency_domain_features(sig.phase_current_1, sig.fs)
print("Frequency-domain features:")
for k, v in fd.items():
    print(f"  {k:<25}: {v:.6f}")

Frequency-domain features:
  spectral_centroid        : 85.554516
  spectral_variance        : 518637.396506
  spectral_std             : 720.164840
  spectral_skewness        : 31.672122
  spectral_kurtosis        : 1101.774745
  total_spectral_energy    : 0.175985
  peak_frequency           : 62.500000
  peak_amplitude           : 1.983097
  energy_0_500Hz           : 0.175714
  energy_500_2000Hz        : 0.000008
  energy_2000_8000Hz       : 0.000032
  energy_8000_16000Hz      : 0.000114


In [264]:
# --- 3c. Wavelet Packet Decomposition Features ---
wpd = wavelet_packet_features(sig.phase_current_1)
print(f"WPD features (first 8 of {len(wpd)}):")
for k, v in list(wpd.items())[:8]:
    print(f"  {k:<25}: {v:.6f}")

WPD features (first 8 of 17):
  energy_band_0            : 703970.866394
  energy_band_1            : 77.554972
  energy_band_2            : 66.528441
  energy_band_3            : 298.172475
  energy_band_4            : 296.275205
  energy_band_5            : 66.296724
  energy_band_6            : 64.751766
  energy_band_7            : 135.434299


In [265]:
# --- 3d. Envelope Analysis (Vibration) ---
# Bandpass → Hilbert → magnitude spectrum; fault frequencies appear as peaks.
env_freqs_vib, env_spec_vib = envelope_analysis(sig.vibration, sig.fs, band=ENVELOPE_BAND)

print("Envelope amplitudes at characteristic frequencies (vibration):")
for name, f_char in char_freqs.items():
    if f_char > 0:
        mask = (env_freqs_vib >= f_char - 2) & (env_freqs_vib <= f_char + 2)
        if np.any(mask):
            peak = np.max(env_spec_vib[mask])
            print(f"  {name:6s} ({f_char:.1f} Hz): {peak:.6f}")

Envelope amplitudes at characteristic frequencies (vibration):
  shaft_freq (15.0 Hz): 0.008646
  BPFO   (45.8 Hz): 0.020081
  BPFI   (74.2 Hz): 0.002278
  BSF    (29.9 Hz): 0.002896
  FTF    (5.7 Hz): 0.010025


In [266]:
# --- 3e. Full Feature Vector ---
all_feats = extract_all_features(sig.phase_current_1, sig.fs, 'current', char_freqs)
print(f"Total features extracted per segment: {len(all_feats)}")

Total features extracted per segment: 57


### 3f. Batch Feature Extraction

Load every bearing folder under `mat_dir`, optionally filtered to `SETTING_FILTER`.
When `SETTING_FILTER = None` all four operating conditions are used, giving
**~15 bearings × 4 conditions × 20 signals = ~1 200 samples**.

**Speed normalisation** is applied after raw feature extraction so that the
feature matrix is condition-invariant:

| Feature group | Raw units | Normalised to |
|---|---|---|
| `fd_spectral_centroid`, `fd_peak_frequency`, `fd_spectral_std` | Hz | orders (÷ f_shaft) |
| `fd_spectral_variance` | Hz² | orders² (÷ f_shaft²) |
| Envelope (`env_*`) | amplitude | *unchanged* — already speed-invariant because they are extracted AT the characteristic frequencies computed from the actual RPM |
| Time-domain, WPD ratios, WPD entropy | dimensionless | *unchanged* |

Runtime: 4–12 minutes depending on hardware (wavelet and envelope steps dominate).

In [267]:
# --- 3f-i. Load all bearing signals (one operating condition) ---
# Iterate only BEARING_SET — never stray into extra folders that may exist on disk.
all_signals: list = []
for bearing_code in BEARING_SET:
    bearing_folder = mat_dir / bearing_code
    if not bearing_folder.is_dir():
        print(f"  WARNING: {bearing_code} not found on disk — skipping")
        continue
    sigs = load_dataset(str(bearing_folder), setting_filter=SETTING_FILTER)
    if N_SIGNALS_PER_BEARING is not None:
        sigs = sigs[:N_SIGNALS_PER_BEARING]
    all_signals.extend(sigs)

print(f"\nLoaded {len(all_signals)} signals  "
      f"({len(BEARING_SET)} bearings x up to {N_SIGNALS_PER_BEARING or 20})")
print("Class distribution:")
for label, name in enumerate(['Healthy', 'OR_damage', 'IR_damage']):
    n = sum(1 for s in all_signals if s.label_3class == label)
    print(f"  {label} -- {name}: {n}")

  Loaded N09_M07_F10_K001_1.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_10.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_11.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_12.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_13.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_14.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_15.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_16.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_17.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_18.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_19.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_2.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_20.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_3.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_4.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_5.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_6.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_7.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_8.mat -> Healthy (K001)
  Loaded N09_M07_F10_K001_9.mat -> Heal

In [ ]:
# --- 3f-ii. Extract features → build X, y, groups matrices ---

def _normalize_freq_features(feats: dict, f_shaft: float) -> dict:
    """Convert absolute-frequency features to speed-invariant orders.

    Dividing Hz features by f_shaft (rpm/60) yields "orders" — multiples of
    the shaft rotation frequency — so the same fault pattern produces the same
    feature value regardless of operating speed.

    Args:
        feats:   Feature dictionary returned by extract_features_from_bearing.
        f_shaft: Shaft rotation frequency in Hz (= rpm / 60).

    Returns:
        Same dict with frequency features replaced by order-domain equivalents.
    """
    # Single-power Hz features (centroid, std, peak frequency)
    hz_suffixes = ('_spectral_centroid', '_spectral_std', '_peak_frequency')
    for k in list(feats):
        if any(k.endswith(s) for s in hz_suffixes):
            feats[k] = feats[k] / (f_shaft + 1e-10)

    # Quadratic Hz features (variance in Hz² → orders²)
    for k in list(feats):
        if k.endswith('_spectral_variance'):
            feats[k] = feats[k] / (f_shaft**2 + 1e-10)

    return feats


feature_list: list = []
label_list:   list = []
groups:       list = []   # bearing code per signal — used for leakage-free splitting

for sig_i in tqdm(all_signals, desc="Extracting features", unit="signal"):
    # Compute bearing-specific fault frequencies from the measured shaft RPM
    rpm = OPERATING_CONDITIONS[sig_i.setting]['speed_rpm']
    char_freqs_i = calc_characteristic_frequencies(rpm)
    f_shaft_i    = rpm / 60   # shaft frequency in Hz

    feats = extract_features_from_bearing(
        sig_i,
        use_current=USE_CURRENT,
        use_vibration=USE_VIBRATION,
        characteristic_freqs=char_freqs_i,
    )

    # Normalise absolute-frequency features to speed-invariant orders
    feats = _normalize_freq_features(feats, f_shaft_i)

    feature_list.append(list(feats.values()))
    label_list.append(sig_i.label_3class)
    groups.append(sig_i.bearing_code)   # track which physical bearing this came from

feature_names = list(feats.keys())          # portfolio variable — CLAUDE.md §8.1
X      = np.array(feature_list, dtype=np.float32)
y      = np.array(label_list,   dtype=np.int64)
groups = np.array(groups)

print(f"Feature matrix : {X.shape}  ({len(feature_names)} features per signal)")
print(f"Label vector   : {y.shape}  classes={np.bincount(y)}")
print(f"Unique bearings: {np.unique(groups)}")

# Encode labels — required portfolio variables (CLAUDE.md §8.1)
label_encoder = LabelEncoder().fit(y)
class_names   = ['Healthy', 'OR_damage', 'IR_damage']
display(
    pd.Series(y)
      .map(dict(enumerate(class_names)))
      .value_counts()
      .rename("count")
      .to_frame()
)

Extracting features:  84%|████████▍ | 1007/1200 [12:38<02:10,  1.48signal/s]

### 3g. DSP Signal Visualisations — Healthy vs Damaged

Each plot shows **K001 (Healthy)** on the left and **KA04 (Outer Race Fault)** on
the right, under identical operating conditions. This side-by-side layout makes
fault signatures immediately visible across all five analysis domains.

In [ ]:
# --- 3h. Load damaged bearing example (KA04 — Outer Race Fault) ---
# KA04 is a real fatigue-induced outer race damage bearing from MINIMAL_SET.
# When SETTING_FILTER is None, pick any available file; otherwise match the filter.
_dmg_pattern = (
    str(mat_dir / 'KA04' / f'*{SETTING_FILTER}*.mat')
    if SETTING_FILTER
    else str(mat_dir / 'KA04' / '*.mat')
)
_dmg_files = glob.glob(_dmg_pattern)
if not _dmg_files:
    raise FileNotFoundError(
        f"No KA04 files found for setting {SETTING_FILTER}. "
        "Check mat_dir or run ensure_data."
    )

sig_dmg = load_mat_file(_dmg_files[0])
char_freqs_dmg = calc_characteristic_frequencies(sig_dmg.speed.mean())

print(f"Damaged bearing  : {sig_dmg.bearing_code}")
print(f"Condition        : {sig_dmg.label_name}  (class {sig_dmg.label_3class})")
print(f"Damage origin    : {sig_dmg.damage_origin}")
print(f"Operating setting: {sig_dmg.setting}")
print(f"Speed            : {sig_dmg.speed.mean():.1f} rpm")
print("\nCharacteristic frequencies:")
for name, f in char_freqs_dmg.items():
    print(f"  {name:6s}: {f:.2f} Hz")

In [ ]:
# --- 3g-i. Time-Domain Signals Comparison ---
_n_show = int(N_SHOW_SECONDS * sig.fs)
_t_ms   = sig.time_64k[:_n_show] * 1000   # ms

_rows = [
    (sig.phase_current_1, sig_dmg.phase_current_1, "Current 1 [A]", C1, D1),
    (sig.phase_current_2, sig_dmg.phase_current_2, "Current 2 [A]", C2, D2),
    (sig.vibration,       sig_dmg.vibration,       "Vibration [g]", C3, D3),
]

fig, axes = plt.subplots(3, 2, figsize=(9, 6.5), sharex=True)
axes[0, 0].set_title(f"Healthy — {sig.bearing_code} ({sig.setting})", fontsize=9)
axes[0, 1].set_title(f"Damaged — {sig_dmg.bearing_code} ({sig_dmg.label_name})", fontsize=9)

for row, (h_sig, d_sig, ylabel, h_col, d_col) in enumerate(_rows):
    axes[row, 0].plot(_t_ms, h_sig[:_n_show], lw=0.5, color=h_col)
    axes[row, 1].plot(_t_ms, d_sig[:_n_show], lw=0.5, color=d_col)
    axes[row, 0].set_ylabel(ylabel, fontsize=8)

for col in range(2):
    axes[2, col].set_xlabel("Time [ms]")

fig.suptitle("Time-Domain Signals Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_time_domain_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-ii. FFT Spectrum Comparison ---
def _fft_db(signal, fs):
    N = len(signal)
    freqs = np.fft.fftfreq(N, 1 / fs)[: N // 2]
    mag   = np.abs(np.fft.fft(signal))[: N // 2] * 2 / N
    return freqs, 20 * np.log10(mag + 1e-10)

_fc_h, _mc_h = _fft_db(sig.phase_current_1,     sig.fs)
_fv_h, _mv_h = _fft_db(sig.vibration,            sig.fs)
_fc_d, _mc_d = _fft_db(sig_dmg.phase_current_1,  sig_dmg.fs)
_fv_d, _mv_d = _fft_db(sig_dmg.vibration,         sig_dmg.fs)

fig, axes = plt.subplots(2, 2, figsize=(9, 5), sharey='row')
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"Damaged — {sig_dmg.bearing_code} ({sig_dmg.label_name})", fontsize=9)

# Row 0: Phase Current 1
for ax, freqs, mag, color, vcolor in [
    (axes[0, 0], _fc_h, _mc_h, C1, C3),
    (axes[0, 1], _fc_d, _mc_d, D1, D3),
]:
    mask = freqs <= FFT_CURRENT_MAX_HZ
    ax.plot(freqs[mask], mag[mask], lw=0.5, color=color)
    ax.axvline(x=100, color=vcolor, ls='--', alpha=0.6, label='100 Hz')
    ax.legend(fontsize=7)

# Row 1: Vibration
for ax, freqs, mag, color, cf, fc in [
    (axes[1, 0], _fv_h, _mv_h, C2, char_freqs,     FAULT_COLORS),
    (axes[1, 1], _fv_d, _mv_d, D2, char_freqs_dmg, FAULT_COLORS_DMG),
]:
    mask = freqs <= FFT_VIB_MAX_HZ
    ax.plot(freqs[mask], mag[mask], lw=0.5, color=color)
    for name, f in cf.items():
        if name in fc:
            ax.axvline(x=f, color=fc[name], ls='--', alpha=0.6, label=name)
    ax.legend(fontsize=6, ncol=2)

axes[0, 0].set_ylabel("Phase Current 1 [dB]", fontsize=8)
axes[1, 0].set_ylabel("Vibration [dB]",        fontsize=8)
for col in range(2):
    axes[1, col].set_xlabel("Frequency [Hz]")

fig.suptitle("FFT Spectrum Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_fft_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-iii. STFT Spectrogram Comparison ---
# Use a fixed-duration segment so STFT and CWT use the same time window.
_STFT_DUR_S = 1.0
_n_stft = int(_STFT_DUR_S * sig.fs)

# Truncate to fixed duration
_c1_h_stft  = sig.phase_current_1[:_n_stft]
_vib_h_stft = sig.vibration[:_n_stft]
_c1_d_stft  = sig_dmg.phase_current_1[:_n_stft]
_vib_d_stft = sig_dmg.vibration[:_n_stft]

# STFT
_f_s, _t_s, _Zch = stft(
    _c1_h_stft,
    fs=sig.fs,
    nperseg=STFT_NPERSEG,
    noverlap=STFT_NOVERLAP
)

_f_v, _t_v, _Zvh = stft(
    _vib_h_stft,
    fs=sig.fs,
    nperseg=STFT_NPERSEG,
    noverlap=STFT_NOVERLAP
)

_, _, _Zcd = stft(
    _c1_d_stft,
    fs=sig_dmg.fs,
    nperseg=STFT_NPERSEG,
    noverlap=STFT_NOVERLAP
)

_, _, _Zvd = stft(
    _vib_d_stft,
    fs=sig_dmg.fs,
    nperseg=STFT_NPERSEG,
    noverlap=STFT_NOVERLAP
)

# Frequency-axis masks
_mf_c = _f_s <= STFT_CURRENT_MAX_HZ
_mf_v = _f_v <= STFT_VIB_MAX_HZ

# Plot
fig, axes = plt.subplots(2, 2, figsize=(9, 6))
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"Damaged — {sig_dmg.bearing_code} ({sig_dmg.label_name})", fontsize=9)

for (ax, t, f, f_mask, Z, cmap) in [
    (axes[0, 0], _t_s, _f_s, _mf_c, _Zch, CMAP),
    (axes[0, 1], _t_s, _f_s, _mf_c, _Zcd, CMAP_DMG),
    (axes[1, 0], _t_v, _f_v, _mf_v, _Zvh, CMAP),
    (axes[1, 1], _t_v, _f_v, _mf_v, _Zvd, CMAP_DMG),
]:
    C = 20 * np.log10(np.abs(Z[f_mask]) + 1e-10)

    # Trim time axis to match C columns (scipy may append one extra boundary frame)
    t_plot = t[:C.shape[1]]

    pcm = ax.pcolormesh(
        t_plot,
        f[f_mask],
        C,
        cmap=cmap,
        shading='gouraud'
    )
    fig.colorbar(pcm, ax=ax, label='dB', pad=0.02)

    # Optional: make the visible x-axis exactly match the fixed-duration window
    ax.set_xlim(0, _STFT_DUR_S)

axes[0, 0].set_ylabel("Phase Current\nFrequency [Hz]", fontsize=8)
axes[1, 0].set_ylabel("Vibration\nFrequency [Hz]", fontsize=8)

for col in range(2):
    axes[1, col].set_xlabel("Time [s]")

fig.suptitle("STFT Spectrogram Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_stft_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-iv. Envelope Spectrum Comparison ---
# Recompute envelopes for both conditions (env_freqs_vib/env_spec_vib from sec3-envelope
# are for sig only; compute damaged and current channel fresh here).
_efv_h, _esv_h = env_freqs_vib, env_spec_vib   # healthy vibration (from sec3-envelope)
_efv_d, _esv_d = envelope_analysis(sig_dmg.vibration,       sig_dmg.fs, band=ENVELOPE_BAND)
_efc_h, _esc_h = envelope_analysis(sig.phase_current_1,     sig.fs,     band=ENVELOPE_CURRENT_BAND)
_efc_d, _esc_d = envelope_analysis(sig_dmg.phase_current_1, sig_dmg.fs, band=ENVELOPE_CURRENT_BAND)

def _env_axes(ax, freqs, spec, cf_dict, fc_dict, color):
    mask = freqs <= ENVELOPE_MAX_HZ
    ax.plot(freqs[mask], 20 * np.log10(spec[mask] + 1e-10), lw=0.8, color=color)
    for name, f in cf_dict.items():
        if name in fc_dict and f <= ENVELOPE_MAX_HZ:
            ax.axvline(x=f, color=fc_dict[name], ls='--', alpha=0.7, label=name)
            for h in [2, 3]:
                if f * h <= ENVELOPE_MAX_HZ:
                    ax.axvline(x=f * h, color=fc_dict[name], ls=':', alpha=0.3)
    ax.legend(fontsize=6, ncol=2)

fig, axes = plt.subplots(2, 2, figsize=(9, 5))
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"Damaged — {sig_dmg.bearing_code} ({sig_dmg.label_name})", fontsize=9)

_env_axes(axes[0, 0], _efv_h, _esv_h, char_freqs,     FAULT_COLORS,     C2)
_env_axes(axes[0, 1], _efv_d, _esv_d, char_freqs_dmg, FAULT_COLORS_DMG, D2)
_env_axes(axes[1, 0], _efc_h, _esc_h, char_freqs,     FAULT_COLORS,     C1)
_env_axes(axes[1, 1], _efc_d, _esc_d, char_freqs_dmg, FAULT_COLORS_DMG, D1)

axes[0, 0].set_ylabel("Vibration [dB]",       fontsize=8)
axes[1, 0].set_ylabel("Phase Current 1 [dB]", fontsize=8)
for col in range(2):
    axes[1, col].set_xlabel("Frequency [Hz]")

fig.suptitle("Envelope Spectrum Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_envelope_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3g-v. CWT Scalogram Comparison ---
# Complex Morlet CWT: same time-frequency principle as STFT but with
# adaptive resolution — narrow time / wide frequency window at high f,
# wide time / narrow frequency window at low f.
_CWT_WAVELET  = 'cmor1.5-1.0'
_CWT_DS       = 8        # downsample 64 kHz → 8 kHz
_CWT_DUR_S    = 1.0
_CWT_N_SCALES = 100
_fs_cwt = sig.fs / _CWT_DS
_dt_cwt = 1.0 / _fs_cwt
_n_cwt  = int(_CWT_DUR_S * _fs_cwt)

_c1_h  = sig.phase_current_1    [: _n_cwt * _CWT_DS : _CWT_DS]
_vib_h = sig.vibration           [: _n_cwt * _CWT_DS : _CWT_DS]
_c1_d  = sig_dmg.phase_current_1 [: _n_cwt * _CWT_DS : _CWT_DS]
_vib_d = sig_dmg.vibration       [: _n_cwt * _CWT_DS : _CWT_DS]
_t_cwt = np.arange(_n_cwt) * _dt_cwt

_f_c_max, _f_v_max, _f_min_cwt = 500, 3000, 5
_sc_c  = np.logspace(np.log10(_fs_cwt / _f_c_max),
                     np.log10(_fs_cwt / _f_min_cwt), _CWT_N_SCALES)
_sc_v  = np.logspace(np.log10(_fs_cwt / _f_v_max),
                     np.log10(_fs_cwt / _f_min_cwt), _CWT_N_SCALES)

_cc_h, _fq_c = pywt.cwt(_c1_h,  _sc_c, _CWT_WAVELET, sampling_period=_dt_cwt)
_cv_h, _fq_v = pywt.cwt(_vib_h, _sc_v, _CWT_WAVELET, sampling_period=_dt_cwt)
_cc_d, _      = pywt.cwt(_c1_d,  _sc_c, _CWT_WAVELET, sampling_period=_dt_cwt)
_cv_d, _      = pywt.cwt(_vib_d, _sc_v, _CWT_WAVELET, sampling_period=_dt_cwt)

fig, axes = plt.subplots(2, 2, figsize=(9, 6))
axes[0, 0].set_title(f"Healthy — {sig.bearing_code}", fontsize=9)
axes[0, 1].set_title(f"Damaged — {sig_dmg.bearing_code} ({sig_dmg.label_name})", fontsize=9)

for (ax, coefs, cmap, ylabel) in [
    (axes[0, 0], _cc_h, CMAP,     "Phase Current\nFrequency [Hz]"),
    (axes[0, 1], _cc_d, CMAP_DMG, None),
    (axes[1, 0], _cv_h, CMAP,     "Vibration\nFrequency [Hz]"),
    (axes[1, 1], _cv_d, CMAP_DMG, None),
]:
    freqs = _fq_c if ax in (axes[0, 0], axes[0, 1]) else _fq_v
    pcm = ax.pcolormesh(_t_cwt, freqs, np.abs(coefs), cmap=cmap, shading='gouraud')
    fig.colorbar(pcm, ax=ax, label='Magnitude', pad=0.02)
    ax.set_yscale('log')
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=8)

for col in range(2):
    axes[1, col].set_xlabel("Time [s]")

fig.suptitle("CWT Scalogram Comparison", fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_cwt_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Model Definition

`TraditionalMLPipeline` (from `03_ml_classification.py`) wraps seven classifiers
(Decision Tree, Random Forest, Gradient Boosting, SVM-RBF, SVM-Linear, kNN, MLP)
plus a majority-vote ensemble, all sharing a `StandardScaler`.
Each model is evaluated identically on the same train/test split, making comparisons fair.

In [ ]:
# =========================================================
# 4. Model Definition
# =========================================================

pipeline_runner = TraditionalMLPipeline()

print("Models registered:")
for name in pipeline_runner.models:
    print(f"  {name}")

## 5. Cross-Validation & Model Selection

`TraditionalMLPipeline.train_and_evaluate` performs an internal stratified split
and trains each classifier. The outer train/test split below (Section 6) serves
as the held-out evaluation set that is not touched during model selection.

In [ ]:
# =========================================================
# 5. Cross-Validation & Model Selection
# =========================================================

# Split by bearing identity, not by signal, to prevent leakage.
# GroupShuffleSplit guarantees that every signal from a given bearing
# lands entirely in train OR test — never split across both.
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Report which physical bearings ended up in each split
train_bearings = np.unique(groups[train_idx])
test_bearings  = np.unique(groups[test_idx])
print(f"Train: {X_train.shape}  bearings={list(train_bearings)}")
print(f"Test : {X_test.shape}   bearings={list(test_bearings)}")
print(f"Class distribution — train: {np.bincount(y_train)}  test: {np.bincount(y_test)}")

results = pipeline_runner.train_and_evaluate(
    X_train, y_train, X_test, y_test,
    feature_names=feature_names,
)

## 6. Final Train / Test Evaluation

Summarise accuracy and F1-score for every model on the held-out test set,
sorted by accuracy descending. The best model is identified for downstream use.

In [ ]:
# =========================================================
# 6. Final Train / Test Evaluation
# =========================================================

print(f"{'Model':<20} {'Accuracy':>10} {'F1 (macro)':>12}")
print("-" * 44)
for name, r in sorted(results.items(), key=lambda x: -x[1]['accuracy']):
    print(f"{name:<20} {r['accuracy']:>10.4f} {r['f1_score']:>12.4f}")

best_model_name = max(results, key=lambda n: results[n]['accuracy'])
print(f"\nBest model: {best_model_name}  "
      f"(accuracy={results[best_model_name]['accuracy']:.4f})")

## 7. Visualisations

Confusion matrices for all eight classifiers on the held-out test set
(n = 60 samples, 20 per class). A perfect classifier produces a diagonal matrix;
off-diagonal entries reveal which fault types are confused.
CART and SVM_Linear are the only models that misclassify any samples —
their errors appear as off-diagonal counts in the grid below.

In [ ]:
# =========================================================
# 7. Visualisations
# =========================================================

# --- 7a. Confusion Matrix Grid (all models) ---
# results[name]['confusion_matrix'] is already computed — no re-training needed.

_model_order = ['CART', 'RF', 'GBT', 'SVM_RBF',
                 'SVM_Linear', 'kNN', 'MLP', 'Ensemble']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for ax, name in zip(axes, _model_order):
    cm = results[name]['confusion_matrix']
    acc = results[name]['accuracy']
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax,
        cbar=False,
        linewidths=0.5,
    )
    ax.set_title(f"{name}  (acc={acc:.3f})", fontsize=9)
    ax.set_xlabel("Predicted", fontsize=8)
    ax.set_ylabel("True", fontsize=8)
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.tick_params(axis='y', labelsize=7, rotation=0)

fig.suptitle("Confusion Matrices — All Models (Test Set, n=60)", fontsize=11)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Explainability

SHAP `TreeExplainer` is used to attribute each prediction to individual features.
`TreeExplainer` works directly with tree-based models (RF, GBT, CART) without
approximation — it computes exact Shapley values in polynomial time.

`shap_vals` has shape `(n_samples, n_features, n_classes)` for multi-class tree models.
The beeswarm summary plot shows the top features by mean |SHAP| across all classes,
revealing which DSP features drive fault discrimination the most.

SHAP is computed on `X_test` only — no training data is touched (no leakage).

In [ ]:
# =========================================================
# 8. Explainability
# =========================================================

# GradientBoostingClassifier (GBT) is not supported by SHAP TreeExplainer for
# multi-class problems — exclude it and prefer RF or CART instead.
_SHAP_UNSUPPORTED = {'GBT'}
_TREE_PRIORITY    = ['RF', 'CART']

shap_model_name = next(
    (m for m in [best_model_name] + _TREE_PRIORITY
     if m in pipeline_runner.models
     and m not in _SHAP_UNSUPPORTED
     and hasattr(pipeline_runner.models[m], 'feature_importances_')),
    None,
)

if shap_model_name is not None:
    fitted_model = pipeline_runner.models[shap_model_name]

    # Use held-out test data only — never training data (no leakage)
    n_shap = min(SHAP_SAMPLE_SIZE, len(X_test))
    X_shap = X_test[:n_shap]

    explainer = shap.TreeExplainer(fitted_model)
    shap_vals = explainer.shap_values(X_shap)   # shape depends on SHAP version

    # Newer SHAP (>=0.40) returns a 3D array (n_samples, n_features, n_classes);
    # older SHAP returns a list of (n_samples, n_features) arrays — handle both.
    if isinstance(shap_vals, np.ndarray) and shap_vals.ndim == 3:
        shap_vals_list = [shap_vals[:, :, i] for i in range(shap_vals.shape[2])]
    else:
        shap_vals_list = shap_vals   # already a list

    # --- 8a. SHAP Summary Plot (all classes, top 20 features) ---
    # SHAP calls color(i) with an integer class index — wrap palette in a callable.
    _palette = np.linspace(0.30, 0.90, len(class_names))
    _shap_color_fn = lambda i: CMAP(_palette[i])

    plt.figure(figsize=FigSize.HEATMAP_LARGE)
    shap.summary_plot(
        shap_vals_list, X_shap,
        feature_names=feature_names,
        class_names=class_names,
        color=_shap_color_fn,
        show=False,
    )
    plt.title(f"SHAP Feature Importance — {shap_model_name} (n={n_shap})")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "06_shap_summary.png", dpi=150, bbox_inches="tight")
    plt.show()

    # --- 8b. Per-class mean |SHAP| table (top 10 features) ---
    shap_df_list = []
    for cls_idx, cls_name in enumerate(class_names):
        vals = np.abs(shap_vals_list[cls_idx])      # (n_shap, n_features)
        mean_abs = vals.mean(axis=0)                # (n_features,)
        shap_df_list.append(
            pd.Series(mean_abs, index=feature_names, name=cls_name)
        )
    shap_summary = pd.DataFrame(shap_df_list).T
    shap_summary['mean_all_classes'] = shap_summary.mean(axis=1)
    shap_summary = shap_summary.sort_values('mean_all_classes', ascending=False)

    print(f"\nTop 10 features by mean |SHAP| ({shap_model_name}):")
    display(shap_summary.head(10).style.format("{:.4f}"))
else:
    print("No compatible tree-based model available for SHAP TreeExplainer; skipping.")

## 9. Summary & Conclusions

**Task:** 3-class bearing fault classification (Healthy / Outer Race Damage / Inner Race
Damage) on the Paderborn University dataset using phase-current (64 kHz) and vibration
(64 kHz) signals.

**Best model:** see Section 6 — CV F1-macro and test F1-macro reported there.

**Key drivers** (from SHAP — Section 8):
- Envelope analysis features at BPFO / BPFI frequencies capture fault-impulse repetition
- RMS and kurtosis of phase current distinguish healthy from damaged bearings
- Wavelet packet sub-band energies differentiate outer-race from inner-race patterns

**Pipeline status:**

| Step | Status |
|---|---|
| Data acquisition (`ensure_data`) | Complete |
| Signal loading & metadata (`load_mat_file`) | Complete |
| DSP feature extraction (time, freq, WPD, envelope) | Complete |
| Signal visualisations (time-domain, FFT, STFT, envelope) | Complete |
| Batch feature extraction from full dataset | Complete |
| ML classification (7 models + ensemble) | Complete |
| SHAP explainability (TreeExplainer) | Complete |

**Limitations & next steps:**
1. Evaluate cross-condition generalisation using `EXPERIMENT_ARTIFICIAL_TO_REAL`
   (train on artificially induced damage, test on real fatigue damage).
2. Add 1D-CNN or STFT-image CNN comparison (`03_ml_classification.py` has the
   architectures ready) — deep features may capture higher-order fault harmonics.
3. Extend `SETTING_FILTER` to all four operating conditions and assess
   speed/load robustness.